# Preprocessing visualisation: progress version
Loads the six selected images with progress, caches them, and reports preprocessing, layout, saving and display stages. Run All using Python (fruit-split).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/preprocess/pipeline.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook inside the fruit-illum-norm repository.')
sys.path.insert(0, str(ROOT))
from src.preprocess.pipeline import PIPELINES
OUTPUT = ROOT / 'outputs/figures'
OUTPUT.mkdir(parents=True, exist_ok=True)
print(ROOT, list(PIPELINES))

## 1. Select sample images
Use three fresh and three rotten training images, with a fixed seed.

In [ ]:
df = pd.read_csv(ROOT / 'data/splits/split_seed42.csv')
assert len(df) == 3200
train = df[df['split'] == 'train']
samples = pd.concat([train[train['label'] == label].sample(3, random_state=42)
                     for label in [0, 1]], ignore_index=True)
samples[['fruit', 'cls', 'path']]

## 2. Load images
Match the dataset loader: RGB, 250 × 250, bilinear resize. Do not apply ImageNet normalisation for display.
Load only the six selected images, with progress and elapsed time. Cache them so both comparison panels reuse the decoded images.

In [ ]:
from functools import lru_cache
from time import perf_counter
from tqdm import tqdm

@lru_cache(maxsize=32)
def load_rgb(path):
    image_path = ROOT / path
    if not image_path.is_file():
        raise FileNotFoundError(f'Sample image not found: {image_path}')
    with Image.open(image_path) as image:
        return np.array(image.convert('RGB').resize((250, 250), Image.Resampling.BILINEAR))

sample_paths = samples['path'].drop_duplicates().tolist()
load_started = perf_counter()
with tqdm(total=len(sample_paths), desc='Loading images', unit='image', file=sys.stdout) as progress:
    for path in sample_paths:
        progress.set_postfix_str(Path(path).name)
        load_rgb(path)
        progress.update(1)
print(f'Images ready: {len(sample_paths)} loaded and cached in {perf_counter() - load_started:.2f}s.', flush=True)

## 3. Compare all five methods
Progress shows preprocessing, layout, PNG export and display. Figures retain 300 dpi; lower PNG compression speeds saving without changing image quality.

In [ ]:
conditions = ['B0', 'B1', 'E1', 'E2', 'E3']
titles = ['Original', 'Gaussian 5×5', 'Gray World', 'CLAHE', 'Gray World + CLAHE']

def comparison_grid(rows, conditions, titles, filename):
    started = perf_counter()
    path = OUTPUT / filename
    fig, axes = plt.subplots(len(rows), len(conditions), figsize=(3.2*len(conditions), 2.9*len(rows)), squeeze=False)
    try:
        with tqdm(total=len(rows)*len(conditions) + 3, desc='Processing panels', unit='step', file=sys.stdout) as progress:
            for i, (_, row) in enumerate(rows.iterrows()):
                rgb = load_rgb(row['path'])
                for j, (condition, title) in enumerate(zip(conditions, titles)):
                    progress.set_postfix_str(f'{Path(row["path"]).name}: {condition}', refresh=False)
                    processed = PIPELINES[condition](rgb.copy())
                    assert processed.shape == rgb.shape and processed.dtype == np.uint8
                    ax = axes[i, j]
                    ax.imshow(processed)
                    ax.set_xticks([])
                    ax.set_yticks([])
                    for spine in ax.spines.values():
                        spine.set_visible(False)
                    if i == 0:
                        ax.set_title(f'{condition} — {title}', fontsize=11)
                    if j == 0:
                        ax.set_ylabel(row['cls'].replace('_', ' '), fontsize=11)
                    progress.update(1)
            progress.set_postfix_str('', refresh=False)
            progress.set_description('Laying out figure')
            fig.tight_layout()
            progress.update(1)
            progress.set_description('Saving PNG')
            fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white', pil_kwargs={'compress_level': 1})
            progress.update(1)
            progress.set_description('Displaying figure')
            plt.show()
            progress.update(1)
            progress.set_description('Done')
    finally:
        plt.close(fig)
    print(f'Saved: {path} ({perf_counter() - started:.2f}s)', flush=True)

comparison_grid(samples, conditions, titles, 'preprocessing_comparison.png')

## 4. Compact panel for presentation
One fresh and one rotten image, showing successive stages of the proposed method.

In [ ]:
comparison_grid(samples.iloc[[0, 3]], ['B0', 'E1', 'E3'],
                ['Original', 'Gray World', 'Gray World + CLAHE'],
                'preprocessing_slide_panel.png')

## 5. Discussion
Inspect smoothing, colour changes, local contrast, and potential colour distortion or texture exaggeration. Visual changes alone do not establish classification improvement. The separate stress experiment will measure robustness.